<a href="https://colab.research.google.com/github/TanmayJain-dev/ArXiv-Paper-Intelligence-System/blob/main/ArXiv_Paper_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Library Installation

In [ ]:
!pip install datasets faiss-cpu sentence_transformers keybert gradio "transformers<5.0.0" "huggingface-hub>=0.34.0,<1.0" --quiet

Dataset Download and Text Cleaning

In [ ]:
import pandas as pd
from datasets import load_dataset

print("Downloading dataset...")
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

df = pd.DataFrame(dataset)
df = df[['title', 'abstract']]
df = df.head(15000)

df['paper_text'] = df['title'] + " " + df['abstract']
df['paper_text'] = df['paper_text'].fillna("").astype(str).str.replace("\n", " ", regex=False).str.strip()

print(f"Data ready! Shape: {df.shape}")

 Smart Vector Cache Controller

In [ ]:
import os
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

index_file = "arxiv_faiss.index"
df_file = "arxiv_data.pkl"

print("Downloading the AI Embedding Model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists(index_file) and os.path.exists(df_file):
    print("\n🚀 Found saved cache! Loading your 15,000 papers in under 3 seconds...")
    index = faiss.read_index(index_file)
    df = pd.read_pickle(df_file)
    print("✅ System assets loaded successfully and ready for the app!")
else:
    print("\n⚠️ No cached files found. Initializing full pipeline...")
    print("Encoding all 15,000 papers... (This might take 1-2 minutes)")
    embeddings = model.encode(df['paper_text'].to_list(), batch_size=32, show_progress_bar=True)

    print("\nBuilding FAISS search index...")
    faiss.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    print("\n💾 Saving assets to disk for future sessions...")
    faiss.write_index(index, index_file)
    df.to_pickle(df_file)
    print("✅ Cache saved successfully!")

Fully Integrated Web Application

In [ ]:
import gradio as gr
from transformers import pipeline
from keybert import KeyBERT

print("Loading core models onto GPU...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)
ner_model = pipeline("ner", aggregation_strategy="simple", device=0)
kw_model = KeyBERT(model=model)

possible_topics = ["Computer Vision", "Natural Language Processing", "Robotics", "Healthcare AI", "Reinforcement Learning"]

def ui_search(query, num_results):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, int(num_results))

    titles, categories, summaries, key_phrases, named_entities = [], [], [], [], []

    for i in range(int(num_results)):
        idx = I[i]
        title_str = df.iloc[idx]['title']
        abstract_text = df.iloc[idx]['abstract']
        clean_abstract = " ".join(abstract_text.split())


        summary = summarizer(clean_abstract, max_length=100, min_length=30, do_sample=False)['summary_text']
        topic = classifier(clean_abstract, candidate_labels=possible_topics)['labels']
        keywords = kw_model.extract_keywords(clean_abstract, top_n=3)
        kw_list = [kw for kw in keywords]

        entities = ner_model(clean_abstract)
        ent_list = list(set([e['word'] for e in entities if e['score'] > 0.80]))
        ent_str = ", ".join(ent_list) if len(ent_list) > 0 else "None"


        titles.append(f"📄 [{i+1}] {title_str}")
        categories.append(f"Paper {i+1}: {topic}")
        key_phrases.append(f"Paper {i+1}: {', '.join(kw_list)}")
        named_entities.append(f"Paper {i+1}: {ent_str}")
        summaries.append(f"📋 Result {i+1}:\n{summary}\n" + ("-"*50 if i < int(num_results)-1 else ""))

    return (
        "\n\n".join(titles),
        "\n\n".join(categories),
        "\n\n".join(key_phrases),
        "\n\n".join(named_entities),
        "\n\n\n".join(summaries)
    )

custom_dark_css = """
body, .gradio-container, html { background-color: #0b0f19 !important; color: #f3f4f6 !important; }
.gradio-container textarea, .gradio-container input { background-color: #111827 !important; color: #ffffff !important; border: 1px solid #374151 !important; }
.gradio-container label, .gradio-container span { color: #f3f4f6 !important; background: transparent !important; }
.gradio-container h1, .gradio-container p, .gradio-container markdown { color: #ffffff !important; }
.gradio-container button.primary { background-color: #ff6600 !important; color: white !important; border: none !important; }
.gradio-container .block { background-color: #111827 !important; border: 1px solid #1f2937 !important; }
"""

with gr.Blocks(theme=gr.themes.Default(primary_hue="orange", neutral_hue="slate"), css=custom_dark_css) as demo:
    gr.Markdown("# 🚀 ArXiv AI Advanced Research Engine")
    gr.Markdown("Powered by FAISS Vector Embeddings, BART Summarizer, KeyBERT, and Token Classifiers via GPU Optimization.")

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(lines=3, placeholder="Ask the assistant a research question...", label="Search Query")
            results_slider = gr.Slider(minimum=1, maximum=5, value=1, step=1, label="Papers to Retrieve")
            submit_btn = gr.Button("Query Knowledge Base", variant="primary")

        with gr.Column(scale=2):
            output_title = gr.Textbox(label="Retrieved Paper Title(s)", interactive=False)
            output_cat = gr.Textbox(label="Predicted Research Topic(s)", interactive=False)
            output_kw = gr.Textbox(label="Extracted Key Phrases (KeyBERT)", interactive=False)
            output_ner = gr.Textbox(label="Detected Entities (NER Model)", interactive=False)
            output_sum = gr.Textbox(label="AI Executive Summary", lines=10, interactive=False)

    submit_btn.click(
        fn=ui_search,
        inputs=[query_input, results_slider],
        outputs=[output_title, output_cat, output_kw, output_ner, output_sum]
    )

demo.launch(share=True)